# Notebook 01 — Whisper Fine-tuning for Indonesian ASR

**Goal:** Fine-tune `openai/whisper-small` on Mozilla Common Voice (Indonesian) to reduce Word Error Rate (WER).

**Pipeline:**
```
Common Voice ID (audio) → Feature Extraction (mel spectrogram)
  → Whisper-small fine-tuned → WER evaluation
  → Save checkpoint to Google Drive
```

**Runtime:** T4 GPU, ~45–60 min for 3 epochs on 2000 samples

## 1. Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate evaluate librosa soundfile jiwer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/indonesian-audio-intelligence'
if not os.path.exists(f'{REPO_DIR}/src'):
    !git clone https://github.com/KimiDandy/indonesian-audio-intelligence {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')

import sys
sys.path.append(REPO_DIR)

CHECKPOINT_DIR = f'{REPO_DIR}/whisper-id-finetuned'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Drive mounted, checkpoint dir ready.')

In [ ]:
import torch
import numpy as np
from dataclasses import dataclass
from typing import Any

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load & Prepare Dataset

In [ ]:
from datasets import load_dataset, Audio

# Google FLEURS Indonesian - public, no login required
# Replaces Mozilla Common Voice 11.0 which was removed from HuggingFace (Oct 2025)

SAMPLE_RATE = 16_000
TRAIN_SAMPLES = 2000
VAL_SAMPLES = 500

print('Loading FLEURS Indonesian dataset...')
cv_train = load_dataset(
    'google/fleurs', 'id_id',
    split=f'train[:{TRAIN_SAMPLES}]',
    trust_remote_code=True
)
cv_val = load_dataset(
    'google/fleurs', 'id_id',
    split=f'validation[:{VAL_SAMPLES}]',
    trust_remote_code=True
)

# Resample to 16kHz in-place
cv_train = cv_train.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
cv_val   = cv_val.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

print(f'Train: {len(cv_train)} samples')
print(f'Val:   {len(cv_val)} samples')
print('Sample:', cv_train[0]['raw_transcription'])

## 3. Load Whisper Processor & Model

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = 'openai/whisper-small'

processor = WhisperProcessor.from_pretrained(MODEL_ID, language='Indonesian', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Force Indonesian language during generation
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='indonesian', task='transcribe'
)
model.config.suppress_tokens = []

total_params = sum(p.numel() for p in model.parameters())
print(f'Whisper-small parameters: {total_params/1e6:.0f}M')

In [ ]:
# ── Baseline WER (before fine-tuning) ──────────────────────────────────────
import evaluate

wer_metric = evaluate.load('wer')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

def transcribe_batch(batch):
    audio_arrays = [x['array'] for x in batch['audio']]
    inputs = processor(audio_arrays, sampling_rate=SAMPLE_RATE, return_tensors='pt', padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        pred_ids = model.generate(**inputs)
    batch['prediction'] = processor.batch_decode(pred_ids, skip_special_tokens=True)
    return batch

# Evaluate on first 100 val samples for speed
val_sample = cv_val.select(range(100))
result = val_sample.map(transcribe_batch, batched=True, batch_size=8)

baseline_wer = wer_metric.compute(
    predictions=result['prediction'],
    references=result['raw_transcription']
)
print(f'Baseline WER (before fine-tuning): {baseline_wer:.4f} ({baseline_wer*100:.1f}%)')

## 4. Fine-tuning with Seq2SeqTrainer

In [ ]:
def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = processor.feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['raw_transcription']).input_ids
    return batch

cv_train_proc = cv_train.map(prepare_dataset, remove_columns=cv_train.column_names, num_proc=2)
cv_val_proc   = cv_val.map(prepare_dataset, remove_columns=cv_val.column_names, num_proc=2)
print('Preprocessing done.')

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Remove BOS token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': round(wer, 4)}

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   # effective batch = 16
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=3,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    logging_steps=25,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=cv_train_proc,
    eval_dataset=cv_val_proc,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer initialized. Starting fine-tuning...')

In [ ]:
import time

start = time.time()
trainer.train()
elapsed = (time.time() - start) / 60
print(f'Training complete in {elapsed:.1f} minutes.')

## 5. Evaluate & Compare WER

In [ ]:
# Re-evaluate on same 100-sample val subset
result_ft = val_sample.map(transcribe_batch, batched=True, batch_size=8)
finetuned_wer = wer_metric.compute(
    predictions=result_ft['prediction'],
    references=result_ft['raw_transcription']
)

print('=' * 45)
print(f'  Baseline WER  : {baseline_wer*100:.1f}%')
print(f'  Fine-tuned WER: {finetuned_wer*100:.1f}%')
print(f'  Improvement   : {(baseline_wer - finetuned_wer)*100:.1f} pp')
print('=' * 45)

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/indonesian-audio-intelligence')
from src.utils.visualize import plot_wer_comparison

fig = plot_wer_comparison(
    model_names=['Whisper-small\n(baseline)', 'Whisper-small\n(fine-tuned)'],
    wer_scores=[baseline_wer, finetuned_wer],
)
fig.savefig('/content/drive/MyDrive/indonesian-audio-intelligence/wer_comparison.png', dpi=150, bbox_inches='tight')
fig.show()

In [ ]:
# Save final model + processor
trainer.save_model(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)
print(f'Model saved to {CHECKPOINT_DIR}')

# Save WER results for README
import json
wer_results = {'baseline_wer': baseline_wer, 'finetuned_wer': finetuned_wer}
with open('/content/drive/MyDrive/indonesian-audio-intelligence/wer_results.json', 'w') as f:
    json.dump(wer_results, f, indent=2)
print('Results saved.')

## 6. Sample Predictions

In [ ]:
print('Sample transcriptions (fine-tuned model):')
print('-' * 60)
for i in range(5):
    ref = cv_val[i]['raw_transcription']
    pred = result_ft[i]['prediction']
    print(f'REF : {ref}')
    print(f'PRED: {pred}')
    print()